In [1]:
import pandas as pd
import numpy as np

In [ ]:
df_1 = pd.read_csv("../data/crime/Stop_Data_2019_to_2022.csv",low_memory=False) ##https://opendata.dc.gov/datasets/1d0cbb1658d54027a54234f6b9bf52bf_45/explore
df_2 = pd.read_csv("../data/crime/Stop_Data.csv",low_memory=False) ##https://opendata.dc.gov/datasets/5c359f8ebdf24d22a1f840d64c5cc540_41/explore

df = pd.concat([df_1,df_2])

## Filtering data to Non-ticket Stops

Previously, MPD released a dataset of just "stop encounters." However, in the most recent dataset, MPD has decided to combine additional interactions, such as routine traffic stops, in this dataset. As a result, the data has fundamentally changed from the previous dataset. In an effort to examine interactions that may be viewed as "stop and frisk," we'll start by examining the intereasctions that didn't result in a ticket (but may have resulted in an arrest) to filter out some of the noise. 

In [ ]:
non_ticket_df = df[df['STOP_TYPE'] == "Non-ticket Stop"]

all_reasons = []

## getting unique list of all stop reasons

for reasons in non_ticket_df['STOP_REASON_NONTICKET'].unique():
    if str(reasons) != 'nan':
        reasons = reasons.replace(';',',')
        reasons_list = reasons.split(',')

        for item in reasons_list:
            if ('Individual' in item and 'character' in item):            
                non_ticket_df.loc[non_ticket_df['STOP_REASON_NONTICKET'] == item,"STOP_REASON_NONTICKET"] = "Individual's characteristics"
                item = "Individual's characteristics"

            elif ('Individual' in item and 'actions' in item):
                non_ticket_df.loc[non_ticket_df['STOP_REASON_NONTICKET'] == item,"STOP_REASON_NONTICKET"] = "Individual's actions"
                item = "Individual's actions"

            if item.strip() not in all_reasons:
                all_reasons.append(item.strip())

Thankfully, MPD provides a reason for the non-ticket stop in the data. This helps us add a context to the interaction. We can see that some of these are not necessarily "stop and frisk," such as responding to a crash or responding to a BOLO/Lookout. For the following analysis, we'll focus on the following subset of data that more or lkess indicates officer discretion in making a stop not directly tied to a call for service:

- Individual's actions
- Individual's characteristics
- Suspicion of criminal activity (self-initiated)

Please not that single stops can be assigned multiple reasons, so we'll start by including all stops that included the reasons above, but may also have other reasons assigned.

In [4]:
for reason in all_reasons:
    sub_df = non_ticket_df[non_ticket_df['STOP_REASON_NONTICKET'].str.contains(reason,regex=False,na=False)]

    eth_df = sub_df['ETHNICITY'].value_counts(normalize=True).reset_index()
    eth_df_counts = sub_df['ETHNICITY'].value_counts().reset_index()

    try:
        print("{} | {} | {}".format(reason, 
                            round(eth_df[eth_df['ETHNICITY'] == 'Black']['proportion'].iloc[0],2),
                            eth_df_counts[eth_df_counts['ETHNICITY'] == 'Black']['count'].iloc[0]))
    except Exception as e:
        continue

Call for service | 0.84 | 66666
Observed a weapon | 0.91 | 1974
Individual's actions | 0.86 | 30352
Traffic violation | 0.86 | 12156
BOLO/Lookout | 0.88 | 18157
Information obtained from witnesses or informants | 0.83 | 7344
Suspicion of criminal activity (self-initiated) | 0.88 | 13949
Warrant/court order | 0.91 | 14450
Individual's characteristics | 0.89 | 8887
Prior knowledge | 0.9 | 4675
Response to crash | 0.79 | 949
Information obtained from law enforcement sources | 0.89 | 9521
Demeanor during a field contact | 0.88 | 4138
Truancy Stop | 1.0 | 1
Medical/Behavioral Health Event | 0.82 | 198
Curfew | 0.86 | 244
Truancy | 0.71 | 1329


In [8]:
non_ticket_vis = non_ticket_df[(non_ticket_df['STOP_REASON_NONTICKET'].str.contains("Individual's",na=False)) | 
                               (non_ticket_df['STOP_REASON_NONTICKET'] == "Suspicion of criminal activity (self-initiated)")]

summary_df = non_ticket_vis.groupby(['ETHNICITY'])['PERSON_SEARCH_PAT_DOWN'].value_counts(normalize=True).reset_index()

summary_df[summary_df['ETHNICITY'].isin(['White','Black','Hispanic/Latino'])]

,ETHNICITY,PERSON_SEARCH_PAT_DOWN,proportion
4,Black,0,0.646599
5,Black,1,0.353401
8,Hispanic/Latino,0,0.870000
9,Hispanic/Latino,1,0.130000
18,White,0,0.858991
19,White,1,0.141009


In [9]:
non_ticket_vis['pat_simple'] = non_ticket_vis['PERSON_PAT_DOWN_REASON'].str.replace(';.*','',regex=True)

for race in ['White','Black','Hispanic/Latino']:
    race_df = non_ticket_vis[non_ticket_vis['ETHNICITY'] == race]

    display(race_df.groupby(['ETHNICITY'])['pat_simple'].value_counts(normalize=True).reset_index())

C:\Users\august.warren\AppData\Local\Temp\ipykernel_40536\758881181.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  non_ticket_vis['pat_simple'] = non_ticket_vis['PERSON_PAT_DOWN_REASON'].str.replace(';.*','',regex=True)


,ETHNICITY,pat_simple,proportion
0,White,Individual<U+0092>s actions,0.293706
1,White,Reasonable Suspicion,0.160839
2,White,Characteristics of an armed individual,0.104895
3,White,Nature of the alleged crime,0.090909
4,White,Other,0.076923
5,White,Consent (consent search only),0.055944
6,White,Exigent Circumstances,0.048951
7,White,Individual<U+0092>s characteristics,0.041958
8,White,Consensual,0.027972
9,White,Exigent circumstances,0.020979


,ETHNICITY,pat_simple,proportion
0,Black,Reasonable Suspicion,0.456038
1,Black,Individual<U+0092>s actions,0.204901
2,Black,Characteristics of an armed individual,0.127463
3,Black,Nature of the alleged crime,0.033729
4,Black,Other,0.029434
5,Black,Individual<U+0092>s characteristics,0.027665
6,Black,Exigent Circumstances,0.025897
7,Black,Consensual,0.019707
8,Black,Consent (consent search only),0.018444
9,Black,Exigent circumstances,0.016422


,ETHNICITY,pat_simple,proportion
0,Hispanic/Latino,Reasonable Suspicion,0.675676
1,Hispanic/Latino,Other,0.121622
2,Hispanic/Latino,Consensual,0.054054
3,Hispanic/Latino,Exigent Circumstances,0.054054
4,Hispanic/Latino,"Other, Reasonable Suspicion",0.027027
5,Hispanic/Latino,Characteristics of an armed individual,0.013514
6,Hispanic/Latino,"Consensual, Exigent Circumstances, Reasonable ...",0.013514
7,Hispanic/Latino,"Exigent Circumstances, Other, Reasonable Suspi...",0.013514
8,Hispanic/Latino,"Exigent Circumstances, Reasonable Suspicion",0.013514
9,Hispanic/Latino,Individuals actions,0.013514


In [10]:
non_ticket_vis['charged'] = np.where(non_ticket_vis['ARREST_CHARGES'].isna(),'No charges','Charged')

non_ticket_vis.groupby('ETHNICITY')['charged'].value_counts(normalize=True,dropna=False)

C:\Users\august.warren\AppData\Local\Temp\ipykernel_40536\1598121817.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  non_ticket_vis['charged'] = np.where(non_ticket_vis['ARREST_CHARGES'].isna(),'No charges','Charged')


ETHNICITY                         charged   
American Indian/Alaska Native     No charges    0.666667
                                  Charged       0.333333
Asian                             No charges    0.560847
                                  Charged       0.439153
Black                             No charges    0.548900
                                  Charged       0.451100
Hispanic                          No charges    0.515473
                                  Charged       0.484527
Hispanic/Latino                   Charged       0.524545
                                  No charges    0.475455
Multiple                          No charges    0.510256
                                  Charged       0.489744
Native Hawaiian/Pacific Islander  No charges    0.625000
                                  Charged       0.375000
Other                             No charges    0.504274
                                  Charged       0.495726
Unknown                           No charge